# Alpha-Ligand — the whole pipeline, in your browser

This notebook runs the three-step **Alpha-Ligand** pipeline on Google's computers.
Nothing is installed on your own machine, and nothing here needs a terminal.

1. **Profile the pocket** — look at the active site of a protein and work out what chemistry a binder would need (charges, greasy patches, rings, hydrogen bonds).
2. **Build molecules** — assemble real 3D molecules that carry that chemistry, with RDKit.
3. **Dock them** — place each molecule inside the pocket with AutoDock Vina and score how well it fits.

**Nothing to edit.** Step 2 gives you text boxes and sliders — type your protein in there and everything else follows from it.

**How to run it:** click the little ▶ play button at the left of each grey code cell, top to bottom, and wait for each one to finish before starting the next. The first cell takes the longest (a few minutes) because it is installing software.

---

### Please read this before you trust any number below

* A Vina score is an **empirical estimate**, not a measurement. It is not a Kd, a Ki or an IC50, and this notebook will never print one.
* Docking scores correlate only **loosely** with real binding affinity. A good score is **not** a validated binder.
* Scores within roughly **1 kcal/mol** of each other should be treated as **tied**, not ranked — that is inside Vina's own error.
* The protein is held **rigid** and all **water molecules are removed**. Real pockets flex and real water often matters.
* The output is a **shortlist to test experimentally**, not a prediction you can take to the bench as fact.

## Step 0 — install the software

This installs RDKit (chemistry), Biopython (protein structures), meeko (prepares molecules for docking) and AutoDock Vina itself.

Vina is not a Python package: it is a single program, so we download the official Linux build from its GitHub release page and put it where the scripts can find it.

Expect this to take **2–5 minutes**. Lots of text will scroll past — that is normal. You only need to run it once per session.

In [ ]:
# --- install the Python libraries -------------------------------------------
# meeko 0.8 needs scipy and gemmi installed explicitly, so they are listed here.
!pip -q install rdkit biopython "meeko>=0.6" scipy gemmi

# --- install AutoDock Vina (a standalone program, not a pip package) --------
VINA_URL = "https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64"
!wget -q -O /usr/local/bin/vina $VINA_URL
!chmod +x /usr/local/bin/vina

# --- check everything actually arrived --------------------------------------
import importlib, shutil, sys

ok = True
for mod in ["rdkit", "Bio", "meeko", "scipy", "gemmi"]:
    try:
        importlib.import_module(mod)
        print("OK   ", mod)
    except Exception as e:
        ok = False
        print("FAIL ", mod, "->", e)

vina_path = shutil.which("vina")
if vina_path:
    print("OK    vina at", vina_path)
    !vina --version
else:
    ok = False
    print("FAIL  vina was not found on the PATH")

print()
print("All dependencies ready." if ok else "Something failed above - re-run this cell before continuing.")

## Step 1 — get the code

This copies the Alpha-Ligand project from GitHub onto this Colab machine, so the three scripts are available to run.

If you run this cell twice you will see *"destination path already exists"* — that is harmless, it just means the code is already here.

In [ ]:
import os

REPO_DIR = "/content/alpha-ligand"

if not os.path.isdir(REPO_DIR):
    !git clone --quiet https://github.com/adamcantcodelol/alpha-ligand.git {REPO_DIR}
else:
    print("Already cloned - pulling the latest version instead.")
    !cd {REPO_DIR} && git pull --quiet

os.chdir(REPO_DIR)
print("Working inside:", os.getcwd())
print()
!ls

## Step 2 — type in your protein and its active site

**You do not need to edit any code.** The cell below shows real text boxes and
sliders. Type into them, then carry on.

> ### The values already in the boxes are only an EXAMPLE
> `3ERT` with residues `A:351,A:383,A:524` is the oestrogen receptor with the
> breast-cancer drug 4-hydroxytamoxifen bound in it. It is there so the notebook
> runs out of the box. **If you want your own protein, overwrite both boxes
> before you run anything.** If you leave them alone, you will get results for
> 3ERT and not for your protein.

What each box means:

* **PDB_CODE** — the 4-character code of a structure in the
  [Protein Data Bank](https://www.rcsb.org/), e.g. `2SHP`. Ignored if you tick
  the upload box.
* **ACTIVE_SITE_RESIDUES** — the residues lining the site you care about,
  written `chain:number` and separated by commas, e.g. `A:459,A:465`.
  *Do not know them? Leave this for a moment, run the next cell (Step 2b) and it
  will suggest some you can copy and paste back in here.*
* **CUTOFF_ANGSTROMS** — how far around those residues counts as "the pocket".
  6 is tight, 8 is generous.
* **NUMBER_OF_MOLECULES / CONFORMERS_PER_MOLECULE** — how many candidate
  molecules to build, and how many 3D shapes of each. Bigger = slower.
* **CONTROL_LIGAND** — a known real binder docked alongside yours as a yardstick.
  `auto` uses one only for the handful of proteins this notebook knows
  (currently only 3ERT); for anything else `auto` means *no control*. Choose
  `custom` and paste a SMILES string if you know a real binder for your pocket.
* **UPLOAD_MY_OWN_FILE** — tick it to upload your own `.pdb` or `.cif` instead
  of using a PDB code. You will get a "Choose files" button when you run the cell.

In [ ]:
#@title  Fill these in, then press the play button  { display-mode: "form" }
#@markdown ### Protein (the defaults are an EXAMPLE - overwrite them)
PDB_CODE = "3ERT" #@param {type:"string"}
ACTIVE_SITE_RESIDUES = "A:351,A:383,A:524" #@param {type:"string"}
#@markdown ### Pocket and molecule settings
CUTOFF_ANGSTROMS = 6 #@param {type:"slider", min:4, max:10, step:0.5}
NUMBER_OF_MOLECULES = 10 #@param {type:"slider", min:3, max:20, step:1}
CONFORMERS_PER_MOLECULE = 10 #@param {type:"slider", min:1, max:20, step:1}
#@markdown ### Control ligand (a known real binder, docked as a yardstick)
CONTROL_LIGAND = "auto" #@param ["auto", "none", "custom"]
CUSTOM_CONTROL_SMILES = "" #@param {type:"string"}
#@markdown ### Or upload your own structure file instead of a PDB code
UPLOAD_MY_OWN_FILE = False #@param {type:"boolean"}

import os

# --- optional upload ---------------------------------------------------------
# STRUCTURE is what the scripts are given: either a 4-character code or a path
# to a file you uploaded. OUT_NAME is the prefix every output file will use.
UPLOADED_FILE = None
if UPLOAD_MY_OWN_FILE:
    from google.colab import files          # Colab-only; ignored elsewhere
    uploaded = files.upload()               # shows a "Choose files" button
    if not uploaded:
        raise SystemExit(
            "You ticked the upload box but did not choose a file.\n"
            "Run this cell again and pick a .pdb or .cif file, or untick the box "
            "and type a PDB code instead.")
    UPLOADED_FILE = list(uploaded.keys())[0]
    print("Uploaded:", UPLOADED_FILE)

if UPLOADED_FILE:
    STRUCTURE = os.path.abspath(UPLOADED_FILE)
    OUT_NAME = os.path.splitext(os.path.basename(UPLOADED_FILE))[0]
else:
    STRUCTURE = PDB_CODE.strip()
    OUT_NAME = STRUCTURE.upper()

# --- which control ligand, if any -------------------------------------------
# Known native ligands, keyed by PDB code. Only used when the code really is
# that protein - a control from a different protein is meaningless, so it is
# never carried over silently.
NATIVE_CONTROLS = {
    "3ERT": ("CC/C(=C(\\c1ccc(O)cc1)c1ccc(OCCN(C)C)cc1)c1ccccc1",
             "4-hydroxytamoxifen_(native)"),
}

CONTROL_SMILES = None
CONTROL_NAME = None
if CONTROL_LIGAND == "custom":
    smi = CUSTOM_CONTROL_SMILES.strip()
    if not smi:
        raise SystemExit(
            "You chose CONTROL_LIGAND = 'custom' but left CUSTOM_CONTROL_SMILES "
            "empty.\nEither paste a SMILES string for a known binder, or set "
            "CONTROL_LIGAND back to 'none'.")
    CONTROL_SMILES, CONTROL_NAME = smi, "custom_control"
elif CONTROL_LIGAND == "auto":
    if UPLOADED_FILE:
        pass                      # an uploaded file has no known native ligand
    elif OUT_NAME in NATIVE_CONTROLS:
        CONTROL_SMILES, CONTROL_NAME = NATIVE_CONTROLS[OUT_NAME]

print("Settings recorded. The next cells will confirm them.")

## Step 2b — not sure which residues to use? Let the structure tell you

Most crystal structures already have something bound in the interesting pocket —
a drug, a substrate analogue, a cofactor. The residues wrapped around that
molecule are a sound first guess at the active site.

This cell looks for bound non-protein molecules (ignoring water, salts and
freezing agents) and prints every amino acid within 5 Å of each, already in the
`chain:number` format the box above wants, so you can copy a line and paste it in.

**This is a suggestion, not an answer.** The computer does not know which pocket
you care about, and a crystal can contain molecules that mean nothing
biologically. Check the list against the paper or the RCSB page for your
structure. If the structure has nothing bound, this cell will say so.

In [ ]:
# Suggest active-site residues from whatever is bound in the crystal.
import os, warnings
warnings.filterwarnings("ignore")
from Bio.PDB import PDBList, PDBParser, MMCIFParser, NeighborSearch, is_aa

NEIGHBOUR_RADIUS = 5.0     # angstroms around the bound molecule
MIN_LIGAND_ATOMS = 6       # ignore tiny fragments

# Things that are almost always crystallisation junk rather than a real ligand.
IGNORED = {
    "HOH", "DOD", "WAT", "SO4", "PO4", "GOL", "EDO", "PEG", "PG4", "1PE",
    "MPD", "ACT", "FMT", "NO3", "NH4", "DMS", "TRS", "EPE", "MES", "CIT",
    "TLA", "CL", "BR", "IOD", "NA", "K", "MG", "CA", "ZN", "MN", "FE",
    "NI", "CU", "CD", "CO", "HG",
}

def _load_structure(source):
    """Load from an uploaded file, or download the PDB code from RCSB."""
    if os.path.isfile(source):
        parser = (MMCIFParser(QUIET=True)
                  if source.lower().endswith((".cif", ".mmcif"))
                  else PDBParser(QUIET=True))
        return parser.get_structure("user", source)
    os.makedirs("output/downloads", exist_ok=True)
    path = PDBList(verbose=False).retrieve_pdb_file(
        source.lower(), pdir="output/downloads", file_format="mmCif")
    if not path or not os.path.isfile(path):
        raise SystemExit(
            "Could not download '%s' from the PDB. Check the 4-character code "
            "and that you are online." % source)
    return MMCIFParser(QUIET=True).get_structure(source, path)

print("Looking inside:", STRUCTURE)
_structure = _load_structure(STRUCTURE)
_model = next(iter(_structure))
_search = NeighborSearch(list(_model.get_atoms()))

_found = []
for _chain in _model:
    for _res in _chain:
        if not _res.get_id()[0].strip():
            continue                       # a normal amino acid or nucleotide
        if is_aa(_res, standard=False):
            continue                       # a modified amino acid, not a ligand
        if _res.get_resname().strip().upper() in IGNORED:
            continue
        if len(list(_res.get_atoms())) < MIN_LIGAND_ATOMS:
            continue
        _near = {}
        for _atom in _res:
            for _nb in _search.search(_atom.get_coord(), NEIGHBOUR_RADIUS):
                _other = _nb.get_parent()
                if _other is _res or not is_aa(_other, standard=True):
                    continue
                _key = "%s:%d" % (_other.get_parent().get_id(),
                                  _other.get_id()[1])
                _near[_key] = _other.get_resname()
        if _near:
            _found.append((_res.get_resname(), _chain.get_id(),
                           _res.get_id()[1], _near))

print()
if not _found:
    print("No bound ligand found in this structure.")
    print()
    print("That is common for apo (empty) structures. The computer cannot guess")
    print("your active site from nothing, so you will have to supply it:")
    print("  * look up the catalytic or binding residues in a paper on this protein,")
    print("  * or open https://www.rcsb.org/structure/%s and read the" % str(OUT_NAME))
    print("    'Binding' / 'Macromolecule' annotations,")
    print("  * or use a structure of the same protein that does have a ligand bound.")
    print("Then type them into ACTIVE_SITE_RESIDUES above as chain:number.")
else:
    print("Found %d bound molecule(s). Residues within %.1f A of each:"
          % (len(_found), NEIGHBOUR_RADIUS))
    for _name, _cid, _num, _near in _found:
        _keys = sorted(_near, key=lambda k: (k.split(":")[0],
                                             int(k.split(":")[1])))
        print()
        print("-" * 70)
        print("Bound molecule %s (chain %s, residue %d) - %d residues nearby"
              % (_name, _cid, _num, len(_keys)))
        print("  with residue names:")
        print("   ", "  ".join("%s(%s)" % (k, _near[k]) for k in _keys))
        print("  COPY THIS into the ACTIVE_SITE_RESIDUES box:")
        print("   ", ",".join(_keys))
    print()
    print("-" * 70)
    print("Tip: you do not have to use the whole list. Three to six residues")
    print("that genuinely line the pocket work well; the cutoff above then")
    print("picks up the rest of the wall around them.")
    print("These are suggestions from the crystal, not a verified active site.")

## Step 2c — check and confirm

This repeats back exactly what is about to be analysed, so a "Run all" never
runs on the wrong protein without you noticing. It also stops here, with a plain
message, if the residues box is empty or written in the wrong format — much
easier to read than an error from deep inside a script.

In [ ]:
# Validate the residues box and echo everything back.
import re, sys

_problems = []
_raw = str(ACTIVE_SITE_RESIDUES).strip()

if not _raw:
    _problems.append(
        "The ACTIVE_SITE_RESIDUES box is empty.\n"
        "    Run Step 2b above, copy one of the suggested lines, paste it into\n"
        "    the box in Step 2, and run Step 2 again.")
else:
    _tokens = [t.strip() for t in _raw.split(",") if t.strip()]
    if not _tokens:
        _problems.append("The ACTIVE_SITE_RESIDUES box has only commas in it.")
    for _t in _tokens:
        if ":" not in _t:
            _problems.append(
                "'%s' has no chain letter. Write it as CHAIN:NUMBER, for "
                "example A:%s." % (_t, _t))
        else:
            _chain, _num = _t.split(":", 1)
            _chain, _num = _chain.strip(), _num.strip()
            if not _chain:
                _problems.append("'%s' is missing the chain letter before the "
                                 "colon (try A:%s)." % (_t, _num))
            if not re.fullmatch(r"-?\d+[A-Za-z]?", _num):
                _problems.append(
                    "'%s' - the part after the colon must be a residue number "
                    "(digits), not '%s'." % (_t, _num))

if not str(PDB_CODE).strip() and not UPLOADED_FILE:
    _problems.append("The PDB_CODE box is empty and no file was uploaded.")
elif not UPLOADED_FILE and len(str(PDB_CODE).strip()) != 4:
    _problems.append(
        "'%s' is not a 4-character PDB code. Codes look like 3ERT or 2SHP."
        % PDB_CODE)

if _problems:
    print("=" * 70)
    print("STOP - something in the form needs fixing")
    print("=" * 70)
    for _p in _problems:
        print("  *", _p)
    print()
    print("Fix the box(es) in Step 2, run Step 2 again, then run this cell again.")
    raise SystemExit("Please fix the form above before continuing.")

ACTIVE_SITE = ",".join(t.strip() for t in _raw.split(",") if t.strip())
CUTOFF = float(CUTOFF_ANGSTROMS)

print("=" * 70)
print("ABOUT TO ANALYSE")
print("=" * 70)
if UPLOADED_FILE:
    print("Structure      : your uploaded file %s" % UPLOADED_FILE)
else:
    print("Structure      : PDB code %s  (https://www.rcsb.org/structure/%s)"
          % (OUT_NAME, OUT_NAME))
print("Active site    : %s" % ACTIVE_SITE)
print("Pocket cutoff  : %.1f A" % CUTOFF)
print("Molecules      : %d, with %d conformers each"
      % (NUMBER_OF_MOLECULES, CONFORMERS_PER_MOLECULE))
if CONTROL_SMILES:
    print("Control ligand : %s" % CONTROL_NAME)
    print("                 %s" % CONTROL_SMILES)
    if CONTROL_LIGAND == "auto":
        print("                 (the known native ligand of %s)" % OUT_NAME)
else:
    print("Control ligand : none")
    if CONTROL_LIGAND == "auto" and not UPLOADED_FILE:
        print("                 (this notebook knows a native ligand only for "
              "%s" % ", ".join(sorted(NATIVE_CONTROLS)) + "; choose 'custom' "
              "above if you know a real binder for this pocket)")
print("Output files   : output/%s_*" % OUT_NAME)
print("=" * 70)
print()
print("If that is not the protein you meant, go back to Step 2, change the "
      "boxes, and run from there again.")

## Step 3 — profile the pocket

This reads the structure (downloading it from RCSB automatically) and describes the pocket: how greasy it is, its net charge, how many hydrogen-bond donors and acceptors line it, how big it is, and — most importantly — a **pharmacophore**: the shopping list of chemical groups a good binder should carry.

**How to read the output:** the ranked feature list near the bottom is the key part. A positively charged wall asks for a *negatively charged* group on the ligand. An aromatic wall asks for a *ring* to stack against. A greasy wall asks for a *greasy* group. That list is what the next step builds molecules to satisfy.

In [ ]:
!python pocket_profiler.py {STRUCTURE} {ACTIVE_SITE} --cutoff {CUTOFF}

print("\n" + "=" * 70)
print("POCKET REPORT")
print("=" * 70)
print(open(f"output/{OUT_NAME}_pocket_report.txt").read())

## Step 4 — build candidate molecules

This takes the pharmacophore from step 3 and assembles molecules that carry those groups, scores them on how well their chemistry covers the pocket's shopping list, and builds the best ones in real 3D with energy-minimised conformers. The result is an `.sdf` file you can open in any molecule viewer.

**How to read the output:** the score out of 100 only says *"does this molecule carry the right chemical groups?"*. It says nothing about whether the molecule physically fits — a top scorer can be far too bulky to get into the pocket. Step 5 is what tests that.

No synthetic-accessibility check is done either, so some of these would be awkward or impossible to actually make.

In [ ]:
!python ligand_builder.py output/{OUT_NAME}_pharmacophore.json --molecules {NUMBER_OF_MOLECULES} --confs {CONFORMERS_PER_MOLECULE}

print("\n" + "=" * 70)
print("MOLECULE REPORT")
print("=" * 70)
print(open(f"output/{OUT_NAME}_ligand_report.txt").read())

## Step 5 — dock the molecules into the pocket

Now the real test. Vina places each molecule inside the pocket in many
orientations and shapes, and reports the best score it can find in kcal/mol
(**more negative = Vina likes it more**).

### The control

A control is a molecule already known to bind this pocket, docked alongside
yours as a yardstick: if the setup is sound, the real binder should score at
least as well as anything we invented. If the control scores badly, do not trust
the other numbers — the setup is wrong, not the chemistry.

You chose the control back in the form in Step 2, and Step 2c printed which one
(if any) is in use. This notebook only knows the native ligand of `3ERT`
(4-hydroxytamoxifen), so for any other protein `auto` means **no control** —
a control from a different protein would be meaningless. If you know a real
binder for your own pocket, set `CONTROL_LIGAND = custom` and paste its SMILES.

This cell is the slow one — roughly **2–6 minutes** for ten molecules.

In [ ]:
# CONTROL_SMILES / CONTROL_NAME were decided by the form in Step 2 and printed
# in Step 2c. Nothing to edit here.
cmd = (
    f"python dock_ligands.py output/{OUT_NAME}_pharmacophore.json "
    f"--ligands output/{OUT_NAME}_ligands.sdf "
    f"--receptor {STRUCTURE} "
    f"--exhaustiveness 8 --poses 5"
)
if CONTROL_SMILES:
    cmd += f' --control-smiles "{CONTROL_SMILES}" --control-name "{CONTROL_NAME}"'
    print("Control ligand in use:", CONTROL_NAME)
else:
    print("No control ligand - scores below can only be compared with each other.")

print("Running:", cmd, "\n")
!{cmd}

print("\n" + "=" * 70)
print("DOCKING REPORT")
print("=" * 70)
print(open(f"output/{OUT_NAME}_docking_report.txt").read())

### Reading the docking scores honestly

* **More negative is better**, but the differences only mean something when they are large. Anything inside about **1 kcal/mol** of another molecule is **tied** — Vina cannot tell them apart.
* Compare your molecules to the **control**. If the real drug is well ahead of everything you generated, your candidates are weaker binders — usually because they are small and do not fill the pocket.
* The receptor was held **rigid** and its **waters were deleted**. Both are approximations that can flip a ranking.
* These scores are **not** affinities. There is no Kd, Ki or IC50 here and you should not convert a score into one.
* What you have is a **shortlist worth testing**, and a reason to test those rather than random molecules. That is all, and it is still genuinely useful.

## Step 6 — make the readable results page

Everything so far has been raw files. This step turns them into **one web page you can
actually read**, plus a spreadsheet.

It writes two files:

* `output/<NAME>_results.html` — the ranked molecules with pictures of their structures,
  their scores in plain English, what the pocket asked for and why, and an honest section on
  what the numbers do *and do not* mean.
* `output/<NAME>_top_hits.csv` — the same table, for Excel or Google Sheets.

**What you do with the HTML file:** double-click it. It opens in any browser like a normal
web page — no Python, no Colab, nothing to install. Everything (styling, every molecule
picture) is embedded *inside* the single file, so it works with no internet and you can
email it to a labmate or your advisor and it will look the same on their machine. It also
prints sensibly if you want it on paper.

The page appears below as soon as the cell finishes, and step 7 downloads it to your computer.


In [ ]:
# Build the readable HTML report + CSV from everything the steps above produced.
# If docking was skipped or failed, this still works - it just writes a
# chemistry-only report and says so on the page.
!python make_report.py output/{OUT_NAME}_pharmacophore.json

import os
from IPython.display import HTML, display

report_path = f"output/{OUT_NAME}_results.html"
csv_path    = f"output/{OUT_NAME}_top_hits.csv"

if os.path.exists(report_path):
    size_kb = os.path.getsize(report_path) / 1024
    print(f"\nReport written: {report_path}  ({size_kb:.0f} KB, self-contained)")
    print(f"Spreadsheet   : {csv_path}")
    print("\nPreview below. Step 7 downloads both files to your computer;")
    print("the HTML one opens in any browser by double-clicking it.\n")

    # Show it inside an iframe so the report's own styling stays in its own box
    # and does not fight with Colab's.
    page = open(report_path, encoding="utf-8").read()
    display(HTML(
        '<iframe style="width:100%; height:900px; border:1px solid #444; '
        'border-radius:8px; background:#0f1115" srcdoc="'
        + page.replace("&", "&amp;").replace('"', "&quot;")
        + '"></iframe>'
    ))
else:
    print("No report was produced - scroll up for the error message.")


## Step 7 — download the results to your computer

This saves the files onto your own machine through your browser. Your browser may ask you to allow multiple downloads — say yes.

| File | What it is |
|---|---|
| `_pocket_report.txt` | the readable pocket description |
| `_pharmacophore.json` | the pocket's chemical shopping list, machine-readable |
| `_ligands.sdf` | the generated molecules in 3D (open in PyMOL, Avogadro, ChimeraX...) |
| `_ligand_report.txt` | the ranked chemistry-only table |
| `_docked.sdf` | the docked **poses** — each molecule positioned inside the pocket |
| `_docking_report.txt` | the final scores, including the control |

To see a pose, open `_docked.sdf` together with the protein structure in a viewer.

The most useful one is **`_results.html`** — that is the readable page. Double-click it
when it lands in your Downloads folder.


In [ ]:
from google.colab import files
import os

wanted = [
    f"output/{OUT_NAME}_pocket_report.txt",
    f"output/{OUT_NAME}_pharmacophore.json",
    f"output/{OUT_NAME}_ligands.sdf",
    f"output/{OUT_NAME}_ligand_report.txt",
    f"output/{OUT_NAME}_docked.sdf",
    f"output/{OUT_NAME}_docking_report.txt",
    f"output/{OUT_NAME}_results.html",      # <- the readable one
    f"output/{OUT_NAME}_top_hits.csv",      # <- opens in Excel
]

for path in wanted:
    if os.path.exists(path):
        print("downloading", path)
        files.download(path)
    else:
        print("skipped (not found):", path)

---

## If something goes wrong

* **It analysed 3ERT when I wanted my own protein** — the boxes in Step 2 still
  had the example in them. Change `PDB_CODE` and `ACTIVE_SITE_RESIDUES`, run
  Step 2 again, then run the cells below it again. Step 2c always prints what is
  about to be analysed, so check that line.
* **"STOP - something in the form needs fixing"** — read the bullet points it
  prints; they say exactly which box is wrong.
* **I do not know my active-site residues** — run Step 2b, which suggests them
  from whatever is bound in the crystal structure.
* **A cell fails with `No such file or directory`** — you probably skipped a
  cell, or an earlier step failed. Run them again in order from the top.
* **`vina: not found`** — re-run the install cell (Step 0). Colab sessions reset
  after a while and lose installed programs.
* **The runtime disconnected** — Colab gives you a free machine and takes it back
  when idle. Reconnect and re-run from Step 0; nothing is lost except time.
* **An error about residues** — check your `chain:number` values really exist in
  that structure. The chain letter matters.

The companion web page does step 1 (pocket profiling) with no notebook at all:
<https://adamcantcodelol.github.io/alpha-ligand/>